In [22]:
import pandas as pd

# 파일별로 모든 컬럼을 불러옴
# usecols: 필요한 열만 읽어와서 메모리를 훨씬 적게 씀

print("데이터를 불러오는 중입니다... 잠시만 기다려주세요.")

# 주문 정보 (340만 건)
orders = pd.read_csv('../data/orders.csv')
# orders = pd.read_csv('../data/orders.csv', usecols = ['order_id', 'user_id', 'eval_set', 'order_number', 'order_hour_of_day', 'days_since_prior_order'])

# 상세 주문 내역 - 과거(3,200만 건)
prior = pd.read_csv('../data/order_products__prior.csv')

# 상세 주문 내역 - 최신(130만 건)
train = pd.read_csv('../data/order_products__train.csv')

# 상품 정보 (5만 건)
products = pd.read_csv('../data/products.csv')

# 상품 진열 위치 정보 (1,000건)
aisles = pd.read_csv('../data/aisles.csv')

# 상품 카테고리 정보 (1,200건)
departments = pd.read_csv('../data/departments.csv')

print("데이터 로드 완료")

데이터를 불러오는 중입니다... 잠시만 기다려주세요.
데이터 로드 완료


In [24]:
# 파일별로 어디에 결측치(NaN)가 있는지 확인
print("--- [파일별 결측치 현황] ---")

# 4개의 데이터프레임을 딕셔너리로 묶음 -> {키, 값} = {파일명, 데이터프레임}
datasets = {'orders': orders, 'prior': prior, 'train': train, 'products': products, 'aisles': aisles, 'departments': departments}

# 각 데이터프레임을 반복하면서 결측치 개수 계산
# items(): 딕셔너리의 키와 값을 동시에 가져올 때 사용 (분리시켜줌)
# missing_count: 시리즈 객체
for name, df in datasets.items():
    missing_count = df.isnull().sum() # 각 열마다 결측치 개수 계산
    print(f"\n--- [{name}] 열 별 결측치 목록 ---") 
    print(missing_count) # 각 열의 결측치 개수 출력
    
    # 결측치가 있는 열이 있다면
    if missing_count.sum() > 0:
        print(f"\n[{name}] 파일 결측치:")
        print(missing_count[missing_count > 0]) # 결측치가 있는 열만 출력
    else:
        print(f"\n--- [{name}] 파일은 결측치가 없습니다. ---")

# 결론: orders의 'days_since_prior_order' 결측치는 첫 주문임을 의미함

--- [파일별 결측치 현황] ---

--- [orders] 열 별 결측치 목록 ---
order_id                       0
user_id                        0
eval_set                       0
order_number                   0
order_dow                      0
order_hour_of_day              0
days_since_prior_order    206209
dtype: int64

[orders] 파일 결측치:
days_since_prior_order    206209
dtype: int64

--- [prior] 열 별 결측치 목록 ---
order_id             0
product_id           0
add_to_cart_order    0
reordered            0
dtype: int64

--- [prior] 파일은 결측치가 없습니다. ---

--- [train] 열 별 결측치 목록 ---
order_id             0
product_id           0
add_to_cart_order    0
reordered            0
dtype: int64

--- [train] 파일은 결측치가 없습니다. ---

--- [products] 열 별 결측치 목록 ---
product_id       0
product_name     0
aisle_id         0
department_id    0
dtype: int64

--- [products] 파일은 결측치가 없습니다. ---

--- [aisles] 열 별 결측치 목록 ---
aisle_id    0
aisle       0
dtype: int64

--- [aisles] 파일은 결측치가 없습니다. ---

--- [departments] 열 별 결측치 목록 ---
department_id    0
d

In [28]:
import pandas as pd

print("--- [데이터 무결성 검사 시작] ---")

# 상세내역(prior, train)에 있는 모든 주문 번호를 하나로 병합
# concat(): 두 표를 위아래로 붙임
all_details = pd.concat([prior['order_id'], train['order_id']])
# print(all_details)

# 상세내역의 주문ID가 주문서(orders)에 포함되어 있는지 확인
# .isin(): 데이터 프레임 객체의 각 요소가 값과 일치하는지 여부를 bool 형식으로 반환
# ~: 부정 연산자
ghost_orders = all_details[~all_details.isin(orders['order_id'])]

if len(ghost_orders) == 0:
    print("성공: 상세내역의 모든 주문ID가 주문서(orders)에 잘 들어있습니다.")
else:
    print(f"경고: 주문서에 존재하지 않는 주문이 {len(ghost_orders)}건 있습니다.")

# 상세내역의 모든 상품ID를 하나로 병합
all_product_ids = pd.concat([prior['product_id'], train['product_id']])

# 상품ID가 상품정보(products) 파일에 등록되어 있는지 확인
unknown_products = all_product_ids[~all_product_ids.isin(products['product_id'])]

if len(unknown_products) == 0:
    print("성공: 모든 상품ID가 상품 정보(products)와 일치합니다.")
else:
    print(f"경고: 정보가 등록되지 않은 상품ID가 {len(unknown_products)}건 발견되었습니다.")

--- [데이터 무결성 검사 시작] ---
성공: 상세내역의 모든 주문ID가 주문서(orders)에 잘 들어있습니다.
성공: 모든 상품ID가 상품 정보(products)와 일치합니다.


[곽세희] 데이터 무결성 및 결측치 체크 결과

1. 파일별 결측치 현황
- orders.csv: days_since_prior_order (약 20만 건) -> 첫 주문 데이터로 판명
- 기타 파일: 결측치 없음 (또는 발견된 수치 기재)

2. 데이터 무결성(ID 매칭) 검사
- order_id 매칭: 주문서(orders)와 상세내역(prior/train) 100% 일치 확인
- product_id 매칭: 상세내역과 상품정보(products) 100% 일치 확인

3. 향후 전처리 전략
- 결측치 채우기 (신규 고객 표시)
  - 아까 찾은 20만 건의 결측치를 단순히 비워두면 AI가 계산을 못함.
  - 해결 방법: days_since_prior_order의 결측치를 -1로 채우거나, is_first_order라는 컬럼을 새로 만들어 1 또는 0을 넣어줌.

- 메모리 최적화 (Downcasting)
  - 지금 데이터가 너무 커서 나중에 모델 돌릴 때 컴퓨터가 멈출 수 있음.
  - 해결 방법: int64(큰 숫자)로 되어 있는 데이터들을 int32나 int16(작은 숫자)으로 변환해서 메모리 용량을 절반 이하로 줄이면 좋을 것 같음.

- 데이터 병합 (Merge)
  - 모델을 학습시키기 위해 6개의 파일을 하나로 합쳐야 함.
  - 해결 방법: orders + order_products + products를 order_id와 product_id 기준으로 조인(Join)하고, **[유저ID | 상품ID | 재구매여부]**가 한 줄에 나오는 학습용 테이블을 만듦
